In [24]:
# ==============================================================================
# Run replay and configured acquisition analysis
# ==============================================================================

import matplotlib.pyplot as plt
import numpy as np
import time

# Visualization style (font sizes intentionally unchanged)
FONT_FAMILY = "sans-serif"
FONT_SIZE_TITLE = 16
FONT_SIZE_LABEL = 14
FONT_SIZE_TICK = 12
FONT_SIZE_LEGEND = 14
FONT_SIZE_ANNOTATION = 12

FIGURE_DPI = 180
FIGSIZE_FIELD = (12, 10)
FIGSIZE_BOXPLOT = (12, 6)
COLORMAP_ACQUISITION = "viridis"

plt.rcParams.update({
    'font.family': FONT_FAMILY,
    'axes.titlesize': FONT_SIZE_TITLE,
    'axes.labelsize': FONT_SIZE_LABEL,
    'xtick.labelsize': FONT_SIZE_TICK,
    'ytick.labelsize': FONT_SIZE_TICK,
    'legend.fontsize': FONT_SIZE_LEGEND,
    'figure.dpi': FIGURE_DPI,
})

In [25]:
from pathlib import Path

# Change this run-data directory and the zero-based iteration to replay another run.
RUN_DATA_DIR = Path(r"C:\Users\user\Desktop\LAB\NH-AL\NH-AL\runs\AK-MCS-EFF\rep_001\reference")
ITERATION_NUMBER = 2
N_MC_SAMPLES = 1_000_000
N_REPETITIONS = 100
RANDOM_SEED = 42
MC_BATCH_SIZE = 4096
PLOT_LIMITS = (-3.5, 3.5)

print(f"Run data: {RUN_DATA_DIR}")
print(f"Iteration: {ITERATION_NUMBER} (zero-based)")
print(f"MC candidates per repetition: {N_MC_SAMPLES:,}; repetitions: {N_REPETITIONS}")


Run data: C:\Users\user\Desktop\LAB\NH-AL\NH-AL\runs\AK-MCS-EFF\rep_001\reference
Iteration: 2 (zero-based)
MC candidates per repetition: 1,000,000; repetitions: 100


In [26]:
import sys

RUN_DATA_DIR = RUN_DATA_DIR.expanduser().resolve()
repo_root = next(
    (parent for parent in RUN_DATA_DIR.parents if (parent / "replay.py").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError(f"Could not locate replay.py above {RUN_DATA_DIR}")
sys.path.insert(0, str(repo_root))
from replay import load_iteration_replay

replay = load_iteration_replay(RUN_DATA_DIR, ITERATION_NUMBER)
problem = replay.problem
model = replay.model
acquisition = replay.acquisition
run_config = replay.run_config
N_TRAIN = len(replay.X_train)
ACQUISITION_LABEL = str(
    run_config["acquisition"].get(
        "type", type(acquisition).__name__.removesuffix("Acquisition")
    )
)
OUTPUT_DIR = RUN_DATA_DIR.parent / f"replay_iteration_{ITERATION_NUMBER:04d}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(
    f"Reconstructed {type(problem).__name__}, {type(model).__name__}, "
    f"and {type(acquisition).__name__} ({acquisition.direction})"
)
print(
    f"Iteration {ITERATION_NUMBER}: n={N_TRAIN}; "
    f"snapshot={replay.evaluation.get('model_snapshot')}"
)


Reconstructed Problem, OrdinaryKriging, and EFFAcquisition (max)
Iteration 2: n=12; snapshot=233f376fa19a8345ae2b


In [27]:
import time
from replay import evaluate_acquisition_mc
from replay_visualization import save_replay_results

print(
    f"\n[Replay] {N_REPETITIONS} repetitions x {N_MC_SAMPLES:,} candidates; "
    f"chunk size={MC_BATCH_SIZE:,}; acquisition={ACQUISITION_LABEL} ({acquisition.direction})"
)
t_start = time.perf_counter()
def report_progress(result):
    repetition = result["repetition"]
    if repetition == 1 or repetition % 10 == 0:
        elapsed = time.perf_counter() - t_start
        print(
            f"  [{repetition:3d}/{N_REPETITIONS}] "
            f"best score={result['acquisition_value']:.3e}; elapsed={elapsed:.1f}s"
        )

results = evaluate_acquisition_mc(
    problem,
    acquisition,
    N_MC_SAMPLES,
    N_REPETITIONS,
    RANDOM_SEED,
    MC_BATCH_SIZE,
    progress_callback=report_progress,
)
total_elapsed = time.perf_counter() - t_start
artifacts = save_replay_results(
    replay,
    results,
    OUTPUT_DIR,
    N_MC_SAMPLES,
    RANDOM_SEED,
    PLOT_LIMITS,
    FIGURE_DPI,
    FIGSIZE_FIELD,
    FIGSIZE_BOXPLOT,
    COLORMAP_ACQUISITION,
    {
        "title": FONT_SIZE_TITLE,
        "label": FONT_SIZE_LABEL,
        "tick": FONT_SIZE_TICK,
        "legend": FONT_SIZE_LEGEND,
        "annotation": FONT_SIZE_ANNOTATION,
    },
    total_elapsed,
)
print(f"\nCompleted in {total_elapsed:.1f}s")
print(f"Saved acquisition field plot: {artifacts['field_plot']}")
print(f"Saved acquisition distribution: {artifacts['distribution_plot']}")
print(f"Saved repetition results: {artifacts['results_csv']}")
print(f"Saved replay summary: {artifacts['summary_json']}")
print(f"{artifacts['summary']['best_acquisition']}")



[Replay] 100 repetitions x 1,000,000 candidates; chunk size=4,096; acquisition=EFF (max)
  [  1/100] best score=6.157e+00; elapsed=0.9s
  [ 10/100] best score=6.196e+00; elapsed=9.0s
  [ 20/100] best score=6.152e+00; elapsed=17.9s
  [ 30/100] best score=6.152e+00; elapsed=26.7s
  [ 40/100] best score=6.152e+00; elapsed=35.5s
  [ 50/100] best score=6.152e+00; elapsed=44.2s
  [ 60/100] best score=6.398e+00; elapsed=53.0s
  [ 70/100] best score=6.415e+00; elapsed=61.9s
  [ 80/100] best score=6.255e+00; elapsed=70.8s
  [ 90/100] best score=6.266e+00; elapsed=79.7s
  [100/100] best score=6.152e+00; elapsed=88.6s

Completed in 88.6s
Saved acquisition field plot: C:\Users\user\Desktop\LAB\NH-AL\NH-AL\runs\AK-MCS-EFF\rep_001\replay_iteration_0002\acquisition_field_plot.png
Saved acquisition distribution: C:\Users\user\Desktop\LAB\NH-AL\NH-AL\runs\AK-MCS-EFF\rep_001\replay_iteration_0002\acquisition_extrema_distribution.png
Saved repetition results: C:\Users\user\Desktop\LAB\NH-AL\NH-AL\runs\A